In [ ]:
import numpy as np
import os
import pytoml
import matplotlib.pyplot as plt

import sys
sys.path.append("../")

from nedflix.physics import Neutrino, InteractionType
from nedflix.conventions import units, Event
from nedflix.source import source_from_config
from nedflix.detector import detector_from_config
from nedflix.fun.sample_event import sample_events

In [ ]:
with open("../resources/configs/powerlaw_source_example.toml") as f:
    source_config = pytoml.load(f)
    source1 = source_from_config(source_config)
    source_config["location"]["right_ascension"] = 200
    source_config["location"]["declination"] = 45
    source2 = source_from_config(source_config)
    
with open("../resources/configs/example_detector.toml") as f:
    detector_config = pytoml.load(f)
    detector_config["response"]["detector_response_file"] = os.path.abspath("../" + detector_config["response"]["detector_response_file"])
    detector = detector_from_config(detector_config)

In [ ]:
events1 = sample_events(detector, source1, 365*units.day)
events2 = sample_events(detector, source2, 365*units.day)

In [ ]:
def is_track(event: Event) -> bool:
    if event.interaction!=InteractionType.ChargedCurrent:
        return False
    if event.initial_neutrino not in [Neutrino.NuMu, Neutrino.NuMuBar]:
        return False
    return True

In [ ]:
colors = ["crimson", "dodgerblue"]
for source, events, color in zip([source1, source2], [events1, events2], colors):
    decs, ras, is_tracks = [], [], []
    for event in events:
        decs = np.append(decs, event.reco_direction.declination)
        ras = np.append(ras, event.reco_direction.right_ascension)
        is_tracks = np.append(is_tracks, is_track(event)).astype(bool)

    plt.scatter(
        np.degrees(ras[~is_tracks]),
        np.degrees(decs[~is_tracks]),
        c=color,
        label="Cascades",
        alpha=0.3
    )
    plt.scatter(
        np.degrees(ras[is_tracks]),
        np.degrees(decs[is_tracks]),
        c=color,
        label="Tracks",
    )
    plt.scatter(
        np.degrees([source.location.right_ascension]),
        np.degrees([source.location.declination]),
        c="k",
        label="Source",
        marker="*"
    )

plt.xlim(0, 360)
plt.ylim(-90, 90)

plt.xlabel(r"Right ascension$~\left[^{\circ}\right]$")
plt.ylabel(r"Declination$~\left[^{\circ}\right]$")

plt.legend()

plt.show()